[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/cnn_fmnist.ipynb)

# Introduction

This notebook presents a simple guide to creating a convolutional neural network with PyTorch. It will predict the outcome of the fashion images from the [Zalando's article images](https://github.com/zalandoresearch/fashion-mnist) (Fashion-MNIST).

The guide contains the most elementary PyTorch elements to create and evaluate a network.
 

In [ ]:
import torch
from torch import nn

from torch.utils.data import DataLoader # loads data in batches
from torchvision import datasets # load Fasion-MNIST
import torchvision.transforms as T # transformers for computer vision 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print('Using {} device'.format(device))

# Torchvision datasets

We obtain the [Fashion-MNIST dataset](http://pytorch.org/vision/main/generated/torchvision.datasets.FashionMNIST.html) via torchvision. The dataset contains a training set of 60,000 examples and a test set of 10,000 examples. Each example is a 28x28 grayscale image, associated with a label from 10 classes. Note that `datasets` is an object imported from torchvision, not to confuse with from the Dataset object (used in torch.utils.data import Dataset)

```from torchvision import datasets```

When called for the first time, the datasets will be downloaded to the path specified in the `root` argument. After that, Torchvision will look first for a local copy before attempting another download.

> **torchvision.transforms**. A transformer operates on the data. Using the ' transform' argument, we can apply multiple transformations (reshape, convert to tensor, normalize, etc.) to the data obtained.

In [ ]:
mytransform = T.ToTensor() # image (3D array) to Tensor

train_data = datasets.FashionMNIST(root = './', download=True, train = True, transform = mytransform)
test_data = datasets.FashionMNIST(root = './', download=True, train = False, transform = mytransform)

Note that the first image in the dataset is a 3D tensor (C, H, W) for the number of channels (C), Height (H), and Width (W).

In [ ]:
image, label = next(iter(train_data))
plt.imshow(image.squeeze(), cmap="gray")
print(label)

In [ ]:
img, label = train_data[0]
img.shape # returns a Tensor of Size 1,28,28

# DataLoader

The PyTorch DataLoader object allows the preparation of the dataset in batches of different sizes and shuffles them if necessary when exposing them to the training. 

```from torch.utils.data import DataLoader```

> Note that the DataLoader object only shuffles the data when `shuffle=True` is given (by default, `shuffle=False`).

## We have 10 classes, as follows

In [ ]:
def output_label(label):
    output_mapping = {
                 0: "T-shirt/Top",
                 1: "Trouser",
                 2: "Pullover",
                 3: "Dress",
                 4: "Coat", 
                 5: "Sandal", 
                 6: "Shirt",
                 7: "Sneaker",
                 8: "Bag",
                 9: "Ankle Boot"
                 }
    input = (label.item() if isinstance(label, torch.Tensor) else label)
    return output_mapping[input]

In [ ]:
demo_loader = DataLoader(train_data, batch_size=10)

batch = next(iter(demo_loader))
images, labels = batch
print(type(images), type(labels))
print(images.shape, labels.shape)

Let's use torchvision functionality to visualize a batch

In [ ]:
import torchvision

grid = torchvision.utils.make_grid(images, nrow=10)

plt.figure(figsize=(15, 20))
plt.imshow(grid.permute(1, 2, 0))  # (C, H, W) -> (H, W, C), as expected by matplotlib
plt.axis("off")
plt.show()
print("labels:", ", ".join(output_label(l) for l in labels))

In [ ]:
torch.manual_seed(101)

train_loader = DataLoader(train_data, batch_size = 100, shuffle=True)
# the test loader can be bigger and doesn't need to be shuffled
test_loader =  DataLoader(test_data,  batch_size = 100, shuffle=False) 

# Create the network (exercise)

Complete the `__init__` method of the class below, defining the layers used in the `forward` method:

*   Make a model class (FashionCNN in our case)
    * It inherits nn.Module class that is a super class for all the neural networks in Pytorch.
*   The Neural Net should have the following layers:
    * Two Sequential layers (`layer1` and `layer2`), each consisting of the following layers:
        * Convolution layer that has kernel size of 3 * 3, padding = 1 (zero_padding) in 1st layer and padding = 0 in second one. Stride of 1 in both layers. 32 filters in the 1st layer and 64 in the second one.
        * Batch Normalization layer.
        * Activation function: ReLU.
        * Max Pooling layer with kernel size of 2 * 2 and stride 2.
    * Flatten out the output for dense layer (a.k.a. fully connected layer).
    * 3 Fully connected layers (`fc1`, `fc2` and `fc3`) with different in/out features (e.g., 600 and 120 hidden units).
    * 1 Dropout layer (`drop`) with probability p = 0.25. Use `nn.Dropout`, since it comes after a fully connected layer (`nn.Dropout2d` drops whole channels of image-like inputs and is meant to be used between convolutional layers).

    * All the functionality is given in the forward method that defines the forward pass of the CNN.
    * Our input image is changing in the following way:
        * First Convolution layer : input: 28 \* 28 \* 1, output: 28 \* 28 \* 32
        * First Max Pooling layer : input: 28 \* 28 \* 32, output: 14 \* 14 \* 32
        * Second Conv layer : input : 14 \* 14 \* 32, output: 12 \* 12 \* 64
        * Second Max Pooling layer : 12 \* 12 \* 64, output:  6 \* 6 \* 64
    * So, the first fully connected layer has 6 \* 6 \* 64 = 2304 input features.
    * Final fully connected layer has 10 output features for 10 types of clothes.

In [ ]:
class FashionCNN(nn.Module):
    
    def __init__(self):
        super().__init__()
        # TODO: define self.layer1, self.layer2 (nn.Sequential),
        #       self.fc1, self.drop, self.fc2 and self.fc3
        
        
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = torch.flatten(out, 1)  # flatten all dimensions except the batch
        out = torch.relu(self.fc1(out))
        out = self.drop(out)
        out = torch.relu(self.fc2(out))
        out = self.fc3(out)
        
        return out

In [ ]:
torch.manual_seed(101)

model = FashionCNN()
model = model.to(device)

* We select the [cross-entropy](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) as the cost function. It measures the difference between the probability distribution predicted for the classes and the true class. Note that `nn.CrossEntropyLoss` applies the softmax internally, so the network outputs raw scores (logits).

* We define the optimization method [Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html) (Adaptive Moment Estimation), a variant of Stochastic Gradient Descent that adapts the learning rate of each parameter.

In [ ]:
learning_rate = 1e-3
error = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training and evaluation

Every 100 iterations (batches), we evaluate the model on the test set. During the evaluation, the model must be put in evaluation mode with `model.eval()` (dropout is disabled and batch normalization uses the statistics learned during training), and then back in training mode with `model.train()`.

In [ ]:
from tqdm import tqdm


def evaluate(model, loader):
    """Returns the accuracy (%) of the model on the data in loader."""
    model.eval()  # evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():  # no need to compute gradients
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            predictions = model(images).argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += len(labels)
    model.train()  # back to training mode
    return 100 * correct / total


num_epochs = 5
eval_every = 100  # iterations between evaluations on the test set
count = 0
accuracy = None  # no evaluation yet
running_loss = 0.0
# Lists for visualization of loss and accuracy
loss_list = []
iteration_list = []
accuracy_list = []

model.train()
for epoch in range(num_epochs):
    loop = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}")
    for images, labels in loop:
        # Transfering images and labels to GPU if available
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = error(outputs, labels)

        # Initializing a gradient as 0 so there is no mixing of gradient among the batches
        optimizer.zero_grad()

        # Propagating the error backward
        loss.backward()

        # Optimizing the parameters
        optimizer.step()

        running_loss += loss.item()
        count += 1

        # Testing the model
        if count % eval_every == 0:
            accuracy = evaluate(model, test_loader)
            loss_list.append(running_loss / eval_every)  # average loss of the last eval_every batches
            iteration_list.append(count)
            accuracy_list.append(accuracy)
            running_loss = 0.0

        # show the loss of the current batch and the last test accuracy in the progress bar
        loop.set_postfix(loss=f"{loss.item():.4f}", test_acc="-" if accuracy is None else f"{accuracy:.2f}%")

    print(f"Epoch {epoch + 1} completed. Iteration: {count}, Loss: {loss_list[-1]:.4f}, Test accuracy: {accuracy:.2f}%")

# Visualization

We'll see the training loss (averaged over the last 100 batches) and the accuracy on the test set along the training iterations.

In [ ]:
plt.plot(iteration_list, loss_list)
plt.xlabel("No. of Iteration")
plt.ylabel("Loss")
plt.title("Iterations vs Loss")
plt.show()

The accuracy on the test set:

In [ ]:
plt.plot(iteration_list, accuracy_list)
plt.xlabel("No. of Iteration")
plt.ylabel("Accuracy")
plt.title("Iterations vs Accuracy")
plt.show()

Finally, we evaluate the final model on all the test data, computing the accuracy for each class, and visualize the accuracy of every outcome for every prediction (i.e., the confusion matrix).

In [ ]:
model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        all_predictions.append(outputs.argmax(dim=1).cpu())
        all_labels.append(labels)

all_predictions = torch.cat(all_predictions)
all_labels = torch.cat(all_labels)

print("Accuracy: {:.2f}%".format((all_predictions == all_labels).float().mean().item() * 100))
for i in range(10):
    mask = all_labels == i
    class_accuracy = (all_predictions[mask] == i).float().mean().item() * 100
    print("Accuracy of {}: {:.2f}%".format(output_label(i), class_accuracy))

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

class_names = [output_label(i) for i in range(10)]

print(classification_report(all_labels, all_predictions, target_names=class_names))
cm = confusion_matrix(all_labels, all_predictions)
cm

In [ ]:
# Show the heatmap corresponding to the confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", xticks_rotation=45)
plt.show()